# Nanopore mapping analysis for PRJEB88111

In [24]:
import polars as pl
import matplotlib.pyplot as plt

In [2]:
depth_df = (
    pl.scan_csv('reps/bams/*.x.all-core.depth.csv')
).collect()

In [3]:
contigs_species_df = pl.read_csv('reps/contigs_to_species.csv')

In [4]:
depth_df

metag,gene,len,hits,breadth,depth_all,depth_cov
str,str,i64,i64,f64,f64,f64
"""ERR15909852""","""FR882202.1""",72839,5938,0.081522,0.088785,1.089087
"""ERR15909852""","""SFNX01000131.1""",4530,0,0.0,0.0,NaN
"""ERR15909852""","""NZ_JAFBIX010000025.1""",46446,8035,0.172997,0.416268,2.406223
"""ERR15909852""","""JALFRH010000010.1""",9875,105,0.010633,0.031899,3.0
"""ERR15909852""","""JALFNN010000087.1""",63285,24763,0.391293,0.56511,1.444211
…,…,…,…,…,…,…
"""ERR15942255""","""SFNX01000182.1""",3218,0,0.0,0.0,NaN
"""ERR15942255""","""JALFNN010000085.1""",4012,0,0.0,0.0,NaN
"""ERR15942255""","""SFNX01000139.1""",3818,0,0.0,0.0,NaN


In [5]:
contigs_species_df.head()

contig,species,acc
str,str,str
"""UREV01000001.1""","""s__Cryptobacteroides sp9005469…","""GCA_900546925.1"""
"""UREV01000002.1""","""s__Cryptobacteroides sp9005469…","""GCA_900546925.1"""
"""UREV01000003.1""","""s__Cryptobacteroides sp9005469…","""GCA_900546925.1"""
"""UREV01000004.1""","""s__Cryptobacteroides sp9005469…","""GCA_900546925.1"""
"""UREV01000005.1""","""s__Cryptobacteroides sp9005469…","""GCA_900546925.1"""


In [6]:
join_df = depth_df.join(contigs_species_df, left_on='gene', right_on='contig', how='inner')
join_df

metag,gene,len,hits,breadth,depth_all,depth_cov,species,acc
str,str,i64,i64,f64,f64,f64,str,str
"""ERR15909852""","""FR882202.1""",72839,5938,0.081522,0.088785,1.089087,"""s__Cryptobacteroides sp0004349…","""GCA_000434935.1"""
"""ERR15909852""","""SFNX01000131.1""",4530,0,0.0,0.0,NaN,"""s__UBA2868 sp004552595""","""GCA_004552595.1"""
"""ERR15909852""","""NZ_JAFBIX010000025.1""",46446,8035,0.172997,0.416268,2.406223,"""s__JAFBIX01 sp021531895""","""GCF_021531895.1"""
"""ERR15909852""","""JALFRH010000010.1""",9875,105,0.010633,0.031899,3.0,"""s__Ornithospirochaeta sp022785…","""GCA_022785155.1"""
"""ERR15909852""","""JALFNN010000087.1""",63285,24763,0.391293,0.56511,1.444211,"""s__Bariatricus sp004560705""","""GCA_022774325.1"""
…,…,…,…,…,…,…,…,…
"""ERR15942255""","""SFNX01000182.1""",3218,0,0.0,0.0,NaN,"""s__UBA2868 sp004552595""","""GCA_004552595.1"""
"""ERR15942255""","""JALFNN010000085.1""",4012,0,0.0,0.0,NaN,"""s__Bariatricus sp004560705""","""GCA_022774325.1"""
"""ERR15942255""","""SFNX01000139.1""",3818,0,0.0,0.0,NaN,"""s__UBA2868 sp004552595""","""GCA_004552595.1"""


In [7]:
join2_df = join_df.with_columns(
    weighted_depth=(pl.col('len') * pl.col('depth_all')),
    weighted_breadth=(pl.col('len') * pl.col('breadth')),
)
join2_df

metag,gene,len,hits,breadth,depth_all,depth_cov,species,acc,weighted_depth,weighted_breadth
str,str,i64,i64,f64,f64,f64,str,str,f64,f64
"""ERR15909852""","""FR882202.1""",72839,5938,0.081522,0.088785,1.089087,"""s__Cryptobacteroides sp0004349…","""GCA_000434935.1""",6467.0,5938.0
"""ERR15909852""","""SFNX01000131.1""",4530,0,0.0,0.0,NaN,"""s__UBA2868 sp004552595""","""GCA_004552595.1""",0.0,0.0
"""ERR15909852""","""NZ_JAFBIX010000025.1""",46446,8035,0.172997,0.416268,2.406223,"""s__JAFBIX01 sp021531895""","""GCF_021531895.1""",19334.0,8035.0
"""ERR15909852""","""JALFRH010000010.1""",9875,105,0.010633,0.031899,3.0,"""s__Ornithospirochaeta sp022785…","""GCA_022785155.1""",315.0,105.0
"""ERR15909852""","""JALFNN010000087.1""",63285,24763,0.391293,0.56511,1.444211,"""s__Bariatricus sp004560705""","""GCA_022774325.1""",35763.0,24763.0
…,…,…,…,…,…,…,…,…,…,…
"""ERR15942255""","""SFNX01000182.1""",3218,0,0.0,0.0,NaN,"""s__UBA2868 sp004552595""","""GCA_004552595.1""",0.0,0.0
"""ERR15942255""","""JALFNN010000085.1""",4012,0,0.0,0.0,NaN,"""s__Bariatricus sp004560705""","""GCA_022774325.1""",0.0,0.0
"""ERR15942255""","""SFNX01000139.1""",3818,0,0.0,0.0,NaN,"""s__UBA2868 sp004552595""","""GCA_004552595.1""",0.0,0.0


In [8]:
sum_df = join2_df.group_by(['metag', 'species']).agg(
    depth_all=pl.col('weighted_depth').sum() / pl.col('len').sum(),
    breadth=pl.col('weighted_breadth').sum() / pl.col('len').sum(),
)
sum_df.sort('breadth')

metag,species,depth_all,breadth
str,str,f64,f64
"""ERR15942159""","""s__Cryptobacteroides sp0004326…",0.0,0.0
"""ERR15941747""","""s__Cryptobacteroides sp0004326…",0.0,0.0
"""ERR15941693""","""s__Cryptobacteroides sp0340892…",0.0,0.0
"""ERR15942217""","""s__Cryptobacteroides sp0004349…",0.0,0.0
"""ERR15942182""","""s__Cryptobacteroides sp0004349…",0.0,0.0
…,…,…,…
"""ERR15935028""","""s__JAFBIX01 sp021531895""",137.764068,0.992922
"""ERR15934940""","""s__Bariatricus sp004560705""",108.242687,0.99321
"""ERR15934725""","""s__Bariatricus sp004560705""",102.132863,0.993625


In [16]:
mf_df = (
    pl.read_csv('PRJEB88111.mf.csv', skip_lines=1)
    .select(["name", "n_hashes"])
)

In [30]:
sum2_df = sum_df.join(mf_df, left_on='metag', right_on='name', how='inner')
sum2_df

metag,species,depth_all,breadth,n_hashes
str,str,f64,f64,i64
"""ERR15938761""","""s__Colivicinus sp002299675""",16.095733,0.980172,4792317
"""ERR15938761""","""s__Colivicinus sp002299675""",16.095733,0.980172,6015297
"""ERR15938761""","""s__Colivicinus sp002299675""",16.095733,0.980172,7952050
"""ERR15942168""","""s__Colivicinus sp002299675""",0.007576,0.000122,88739
"""ERR15942168""","""s__Colivicinus sp002299675""",0.007576,0.000122,120320
…,…,…,…,…
"""ERR15936647""","""s__Colivicinus sp002299675""",6.362538,0.96648,2764116
"""ERR15936647""","""s__Colivicinus sp002299675""",6.362538,0.96648,3293610
"""ERR15942126""","""s__Floccifex porci""",1.153239,0.008441,94604


In [29]:
filt_df = sum2_df.filter(pl.col('n_hashes') > 4e6)
filt_df

metag,species,depth_all,breadth,n_hashes
str,str,f64,f64,i64
"""ERR15938761""","""s__Colivicinus sp002299675""",16.095733,0.980172,4792317
"""ERR15938761""","""s__Colivicinus sp002299675""",16.095733,0.980172,6015297
"""ERR15938761""","""s__Colivicinus sp002299675""",16.095733,0.980172,7952050
"""ERR15935141""","""s__Prevotella sp000434975""",79.086944,0.933354,6369270
"""ERR15935141""","""s__Prevotella sp000434975""",79.086944,0.933354,7920196
…,…,…,…,…
"""ERR15934506""","""s__Prevotella sp000434975""",76.334309,0.924323,7079510
"""ERR15934506""","""s__Prevotella sp000434975""",76.334309,0.924323,9149952
"""ERR15934877""","""s__Cryptobacteroides sp0004349…",4.72089,0.800611,5579319


In [44]:
by_species = filt_df.group_by('species').agg(
    min_breadth=pl.col('breadth').min().round(3),
    med_breadth=pl.col('breadth').median().round(3),
    max_breadth=pl.col('breadth').max().round(3),
    min_depth=pl.col('depth_all').min().round(2),
    med_depth=pl.col('depth_all').median().round(1),
    max_depth=pl.col('depth_all').max().round(1),
)

In [45]:
with pl.Config(tbl_rows=-1):
    print(by_species.sort('min_breadth'))

shape: (16, 7)
┌────────────────────┬─────────────┬─────────────┬─────────────┬───────────┬───────────┬───────────┐
│ species            ┆ min_breadth ┆ med_breadth ┆ max_breadth ┆ min_depth ┆ med_depth ┆ max_depth │
│ ---                ┆ ---         ┆ ---         ┆ ---         ┆ ---       ┆ ---       ┆ ---       │
│ str                ┆ f64         ┆ f64         ┆ f64         ┆ f64       ┆ f64       ┆ f64       │
╞════════════════════╪═════════════╪═════════════╪═════════════╪═══════════╪═══════════╪═══════════╡
│ s__UBA2868         ┆ 0.33        ┆ 0.798       ┆ 0.929       ┆ 5.66      ┆ 19.6      ┆ 52.3      │
│ sp004552595        ┆             ┆             ┆             ┆           ┆           ┆           │
│ s__Floccifex porci ┆ 0.45        ┆ 0.944       ┆ 0.97        ┆ 11.98     ┆ 31.4      ┆ 67.1      │
│ s__Mogibacterium_A ┆ 0.484       ┆ 0.963       ┆ 0.991       ┆ 8.89      ┆ 24.9      ┆ 80.4      │
│ kristiansen…       ┆             ┆             ┆             ┆           ┆